# Clinical AI/ML Portfolio Project: End-to-End Workflow

**Planned system:** public clinical/health dataset → reproducible risk model → versioned API → web application → Android application → CI/CD and monitored deployment.

**Status:** design phase only. No dataset has been downloaded and no model or application has been implemented.

**Portfolio objective:** demonstrate sound clinical framing, reproducible data science, calibrated and responsibly evaluated ML, production engineering, security, deployment, documentation, and software/data licensing.

> Educational/research portfolio only. The eventual application must not diagnose, treat, or replace professional medical judgment. It must not be presented as a validated medical device.

## 1. Recommended project definition

Build a **cardiometabolic risk screening demonstrator** using public-use NHANES data. A suitable first prediction task is a clearly defined binary outcome such as diabetes status, using only variables that would be available at the stated prediction time. Final target and feature definitions must be approved after inspecting the selected cycle codebooks.

Why this is a strong first project:

- NHANES combines demographics, questionnaires, examinations, and laboratory measurements.
- Public-use files and documentation are downloadable from the CDC, with identifiers removed.
- The complex survey design creates an opportunity to show correct use of sample weights, strata, and primary sampling units.
- The project supports interpretable baseline models, calibration, subgroup analysis, an API, and user-facing clients.

Important framing choice: decide whether the model estimates **current screening status** or **future risk**. Cross-sectional NHANES data can support the former but generally cannot justify claims of prospective future-risk prediction. Never label a contemporaneous classifier as a future-risk model.

## 2. Dataset decision gate

| Candidate | Access | Best use | Main constraint | Decision |
|---|---|---|---|---|
| CDC NHANES public-use data | Direct public download | Population screening model with exams/labs/questionnaires | Complex survey analysis; mostly cross-sectional | **Recommended** |
| MIMIC-IV demo | Direct small demo download | Pipeline/schema prototype with realistic EHR tables | Small and not suitable for a credible final clinical model | Optional integration test |
| Full MIMIC-IV | Credentialed access, training, and DUA | Hospital/ICU prediction | Data cannot be redistributed; access burden | Later extension only |
| Curated toy datasets | Usually direct download | Fast UI/API prototype | Often weak provenance, small sample, unrealistic evaluation | Avoid as final evidence |

Before download, record in `docs/data_source_decision.md`: source owner, stable URL/DOI, version or survey cycle, access date, exact files, checksums, license/terms, citation, redistribution rules, intended population, outcome availability, and known limitations.

Authoritative starting points:

- [CDC NHANES datasets and documentation](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/)
- [CDC overview of NHANES public and restricted data](https://www.cdc.gov/nchs/nhanes/about/index.html)
- [CDC NHANES dataset tutorial](https://wwwn.cdc.gov/nchs/nhanes/tutorials/datasets.aspx/1000)
- [PhysioNet MIMIC-IV access terms](https://physionet.org/content/mimiciv/)

Do not commit raw data to Git. Download it through a versioned script into an ignored local directory or an access-controlled object store.

## 3. System architecture

```text
Official dataset source
        │ versioned download + checksum
        ▼
Raw (immutable) → validated/interim → curated features
                                      │
                         train/evaluate/package
                                      ▼
                           versioned model artifact
                                      │
                              FastAPI service
                           ┌──────────┴──────────┐
                           ▼                     ▼
                    Web application       Android application
                           └──────────┬──────────┘
                                      ▼
                      logs, metrics, alerts, audit trail
```

The API owns input validation, feature transformation, model invocation, response schema, model version, and safety messages. Web and Android clients must not independently reimplement model logic.

## 4. Proposed monorepo layout

```text
Clinical/
├── .github/workflows/       # CI, security scans, build, release/deploy
├── android/                 # Kotlin + Jetpack Compose client
├── api/                     # FastAPI service and API tests
├── web/                     # responsive web client
├── src/clinical_ml/         # reusable ingestion, features, training, evaluation
├── notebooks/               # exploration and reports; this file starts the series
├── tests/                   # unit, integration, contract, and data tests
├── data/                    # ignored raw/interim data; small schemas only in Git
├── models/                  # ignored binaries or model registry pointers
├── configs/                 # versioned data/model/application configuration
├── docs/                    # data card, model card, architecture, threat model
├── infrastructure/          # containers and deployment configuration
├── LICENSE                  # license for original repository code
├── NOTICE                   # third-party attribution and data notices
├── CITATION.cff             # preferred citation
├── SECURITY.md
├── CONTRIBUTING.md
├── README.md
└── .gitignore
```

One repository makes cross-client API contract changes easy to review. Components can be split later only when release ownership requires it.

## 5. Step-by-step delivery method

### Phase 0 — Governance and clinical question

1. Write a one-sentence intended-use statement and an explicit non-use statement.
2. Specify target population, care/context setting, prediction time, outcome, prediction horizon (if any), and intended user.
3. Draw the causal/time ordering of every proposed feature and the outcome to prevent leakage.
4. Define benefits, foreseeable harms, unacceptable failure modes, and escalation language.
5. Decide success thresholds before training: discrimination, calibration, sensitivity/specificity at a justified threshold, subgroup performance, latency, uptime, and accessibility.

**Exit gate:** a reviewer can tell exactly what the score means, what it does not mean, and when it may be computed.

### Phase 1 — Reproducible project foundation

1. Initialize Git with protected `main`, short feature branches, pull requests, and conventional commits.
2. Pin Python and JavaScript/Kotlin dependencies; add formatters, linters, type checks, pre-commit hooks, and secret scanning.
3. Add issue/PR templates and a Definition of Done.
4. Capture environment creation and one-command local startup in the README.
5. Tag releases with semantic versions and generate a changelog.

**Exit gate:** a clean clone can run the test skeleton in a documented, pinned environment.

### Phase 2 — Data acquisition and validation

1. Select and freeze one or more compatible NHANES cycles; do not silently mix incompatible variables or weights.
2. Implement a download manifest with official URLs, file versions, expected SHA-256 checksums, and retrieval date.
3. Preserve raw files unchanged. Transform raw → interim → curated through idempotent scripts.
4. Join component files using the documented participant key (`SEQN`) and validate join cardinality.
5. Validate schema, ranges, units, missing/special codes, duplicates, and row counts. Keep missingness semantically distinct from zero.
6. Create a data dictionary and cohort flow diagram with inclusion/exclusion counts.
7. Incorporate NHANES weights, strata, and clusters where the estimand requires population-representative analysis.

**Exit gate:** the analytical cohort can be rebuilt from source files by one tested command, and every exclusion is counted.

### Phase 3 — EDA and split design

1. Inspect prevalence, distributions, plausible ranges, missingness, correlations, and subgroup representation.
2. Define train/validation/test partitioning before feature fitting. Prefer cycle-based temporal/external validation if compatible cycles are available; otherwise use stratified participant-level splits and clearly label the limitation.
3. Fit imputers, scalers, encoders, and feature selectors on training data only. Package them with the estimator as one pipeline.
4. Do not use an outcome-defining measurement as a predictor for that same outcome unless the task is explicitly reconstruction.

**Exit gate:** an EDA report documents leakage checks, split rationale, missingness strategy, and limitations.

### Phase 4 — Modeling and responsible evaluation

1. Establish a prevalence-only baseline and a regularized logistic-regression baseline.
2. Compare a small number of justified candidates (for example, logistic regression and gradient-boosted trees) using identical splits and preprocessing rules.
3. Tune only on training/validation data; evaluate the locked model once on the held-out test set.
4. Report ROC-AUC and PR-AUC, but emphasize calibration curve, Brier score, sensitivity, specificity, PPV, NPV, and confidence intervals. Metrics must match the intended use and prevalence.
5. Select any operating threshold from an explicit harm/benefit rationale—not from accuracy alone.
6. Evaluate performance and calibration across clinically relevant age/sex/race-ethnicity and other available groups. Treat small subgroup estimates cautiously and report sample sizes and uncertainty.
7. Inspect failure cases and perform sensitivity analyses for missingness, alternate definitions, and survey cycles.
8. Prefer transparent feature contribution explanations; state clearly that feature importance is not causality.
9. Produce a model card recording version, training data, intended use, metrics, threshold, limitations, ethical considerations, and reproducibility information.

**Exit gate:** the selected model beats the preregistered baseline, is acceptably calibrated, has no hidden leakage, and has documented subgroup behavior and limitations.

### Phase 5 — Model packaging and API

1. Serialize the complete preprocessing-plus-model pipeline with its schema, feature order, model version, training-data fingerprint, and checksum.
2. Build a FastAPI service with `/health`, `/ready`, `/metadata`, and versioned `/v1/predict` endpoints.
3. Validate types, units, ranges, required/optional fields, and cross-field constraints. Reject invalid inputs with useful errors.
4. Return probability/score, threshold category if justified, model version, input warnings, and concise safety text. Do not return a diagnosis.
5. Generate an OpenAPI specification and use it as the contract for web and Android clients.
6. Add unit, schema, contract, integration, concurrency, and latency tests. Include fixed golden predictions with tolerances.
7. Containerize with a non-root user, minimal base image, health checks, pinned dependencies, and no embedded secrets.

**Exit gate:** the container passes tests and security scans; clients can be generated/tested against the versioned contract.

### Phase 6 — Web application

1. Build a responsive, accessible form driven by the API schema. Display units and allowed ranges next to every field.
2. Explain results using plain language, uncertainty, limitations, and next-step guidance that directs medical concerns to qualified professionals.
3. Include loading, validation, empty, timeout, server-error, and offline states.
4. Meet WCAG-oriented keyboard, contrast, focus, label, and screen-reader checks.
5. Avoid storing health inputs by default. If telemetry is enabled, collect no raw clinical values and obtain appropriate consent.

**Exit gate:** end-to-end tests cover valid, invalid, boundary, API-failure, and accessibility paths.

### Phase 7 — Android application

1. Build a Kotlin/Jetpack Compose client using a generated or contract-tested API layer.
2. Use HTTPS only, environment-specific base URLs, network security configuration, and no API secrets in the APK.
3. Mirror the web flow and safety language without duplicating risk computation locally.
4. Handle rotation/process recreation, accessibility, slow networks, cancellation, retries, and offline status.
5. Add unit tests, Compose UI tests, API mock tests, and at least one emulator end-to-end smoke test.
6. Configure signed release builds through protected CI secrets; never commit signing keys.

**Exit gate:** debug and release candidates build reproducibly, pass automated tests, and communicate with the staging API.

### Phase 8 — Deployment, observability, and operations

1. Create separate development, staging, and production environments. Promote the same immutable image/artifact between them.
2. Use infrastructure as code, a secret manager, least-privilege service identities, TLS, rate limiting, request-size limits, and dependency/image scanning.
3. Log request IDs, model version, latency, status, and coarse validation outcomes—never raw health inputs, tokens, or identifiers.
4. Monitor availability, latency, error rates, schema violations, score distribution, missingness, and drift proxies. Clinical performance cannot be inferred from production predictions without verified outcomes.
5. Define alerts, rollback, incident response, retention, model deprecation, and kill-switch procedures.
6. Publish a reproducible demo using synthetic example inputs; do not host source participant rows.

**Exit gate:** a staged release, smoke test, observability check, rollback rehearsal, and release notes all succeed.

## 6. CI/CD design

| Trigger | Required checks | Result |
|---|---|---|
| Every pull request | formatting, lint, types, unit tests, notebook syntax, data-contract tests, API contract, web tests, Android unit tests, secret/license/dependency scan | Merge blocked on failure |
| Merge to `main` | all PR checks, integration tests, build API/web/Android artifacts, create SBOM, scan containers | Versioned staging artifacts |
| Staging deployment | migrations/config validation, smoke tests, end-to-end tests, latency and accessibility checks | Candidate eligible for approval |
| Version tag/release | reproducibility checks, signed artifacts/provenance, approval gate | Production deployment/release |
| Scheduled | dependency audit, stale-data/source check, drift report, disaster-recovery check | Issue/alert on failure |

Deployment principles: immutable artifacts, least-privilege credentials, protected environments, concurrency control, automatic rollback on failed health checks, and documented manual rollback. Model promotion is separate from code merge and requires evaluation evidence.

## 7. Version control, reproducibility, and artifact lineage

- **Git:** source, configuration, schemas, tests, documentation, and small metadata manifests.
- **Not Git:** raw/derived participant data, model binaries, secrets, signing keys, local databases, and build outputs.
- **Data version:** official cycle/version + exact file list + checksum manifest + transformation commit.
- **Experiment record:** Git commit, configuration, random seed, environment lockfile, data fingerprint, metrics, plots, and artifact checksum.
- **Model version:** semantic or registry version linked to experiment and immutable artifact.
- **API version:** explicit contract path such as `/v1`; breaking changes require a new major API version and deprecation window.
- **Release version:** annotated Git tag, changelog, SBOM, signed build provenance, and rollback target.

Use fixed seeds where supported, but do not claim perfect determinism across different hardware or library versions unless verified. A reproducibility test should rebuild the curated cohort and reproduce metrics within declared tolerances.

## 8. Copyright, licensing, attribution, and privacy

1. Choose a repository code license deliberately (for example Apache-2.0 or MIT) only for original code you have the right to license. Add the full license text and a copyright notice with owner and year.
2. Data are governed separately. Preserve the source's citation, terms, version, and redistribution restrictions in `NOTICE` and the data card. A software license does not relicense the dataset.
3. Maintain a third-party dependency inventory and Software Bill of Materials. Check license compatibility for Python, web, Android, fonts, icons, model artifacts, and copied snippets.
4. Never copy text, figures, UI assets, or code merely because they are visible online. Use original work or material whose license permits the intended use; retain required attribution.
5. Add `CITATION.cff`, references for data/methods, and an acknowledgments section. Do not use CDC, hospital, university, or vendor names/logos in a way that implies endorsement.
6. Add Terms/Disclaimer and Privacy documentation for the deployed demo. State what is transmitted, logged, retained, and deleted. Avoid collecting identifiers or health inputs; public de-identified source data does not automatically make new user-entered health data harmless.
7. Do not call the portfolio HIPAA-compliant, clinically validated, FDA-cleared, or suitable for patient care without the necessary legal, regulatory, security, and clinical evidence.

This is an engineering checklist, not legal advice. Re-check the exact dataset terms and dependency licenses at the time of use.

## 9. Verification matrix

| Layer | Core verification |
|---|---|
| Data | checksums, schema/range/unit tests, join cardinality, cohort counts, leakage review |
| ML | baseline comparison, locked test set, calibration, uncertainty, subgroup analysis, sensitivity analysis |
| API | unit/schema/contract/integration/load tests, golden predictions, malformed-input security tests |
| Web | component and end-to-end tests, responsive layout, accessibility, failure states |
| Android | unit/UI/API-mock/emulator tests, lifecycle and network-failure handling |
| Supply chain | dependency and container scans, secret scan, SBOM, artifact signing/provenance |
| Deployment | staging smoke/E2E tests, observability, rollback and kill-switch rehearsal |
| Documentation | README runbook, data card, model card, API docs, architecture, threat model, license/notice |

No single metric is a release gate. A model may have good AUC and still fail because of poor calibration, leakage, unsafe framing, subgroup instability, or unreproducible data processing.

## 10. Milestones and portfolio demonstrations

1. **M0 — Approved design:** intended use, dataset decision, architecture, governance checklist.
2. **M1 — Reproducible cohort:** download/validation pipeline, data dictionary, cohort diagram, EDA.
3. **M2 — Evaluated model:** baseline comparison, locked test results, calibration/fairness report, model card.
4. **M3 — Production API:** container, OpenAPI contract, tests, model/version metadata, security controls.
5. **M4 — Web demo:** accessible responsive interface, safe result communication, end-to-end tests.
6. **M5 — Android demo:** Compose application, staging integration, signed release candidate.
7. **M6 — CI/CD and public release:** automated pipelines, hosted demo, observability, SBOM, documentation, demo video.

Each milestone should produce a reviewable artifact and a short decision record, not just code.

## 11. Decisions required before implementation

The next step should settle these items before downloading data:

- Exact task: current diabetes screening status, another cardiometabolic outcome, or a truly longitudinal outcome.
- Intended user and input setting: self-service educational demo, student/researcher tool, or clinician-facing prototype.
- Feature tier: questionnaire-only, non-invasive measurements, or laboratory-inclusive. Separate models may be clearer than silently mixing availability.
- NHANES cycle(s) and compatibility of outcome/features across cycles.
- Deployment target and expected operating cost.
- Code license and copyright owner name.

**Recommended next action:** write the intended-use statement and perform the dataset/variable feasibility audit. Implementation should begin only after that gate is approved.